In [38]:
import pandas as pd
import re

file_path = r"C:\Users\husnu\Downloads\клиент_ромашка_продажи.xlsx"   # kendi dosya adını yaz

# tarih + номер накладной ile başlayan satırları yakala
invoice_pattern = re.compile(r"^\d{2}/\d{2}/\d{4}\s+номер накладной", re.IGNORECASE)

# ürün kodu satırı
code_pattern = re.compile(r"^код продукта\s*-\s*(.+)$", re.IGNORECASE)

def parse_sheet(df, sheet_name):
    records = []

    current_product = None
    current_product_code = None
    current_supplier = None

    for idx, row in df.iterrows():
        col_a = row.iloc[0]

        if pd.isna(col_a):
            continue

        text = str(col_a).strip()

        # ürün adı
        if text.lower().startswith("продукт"):
            current_product = text
            continue

        # ürün kodu
        code_match = code_pattern.match(text)
        if code_match:
            current_product_code = code_match.group(1).strip()
            continue

        # tedarikçi
        if text.lower().startswith("поставщик"):
            current_supplier = text
            continue

        # sadece invoice satırlarını al
        if invoice_pattern.match(text):
            # gerekiyorsa tarih ve накладной numarasını ayır
            date_match = re.match(r"^(\d{2}/\d{2}/\d{4})", text)
            invoice_date = date_match.group(1) if date_match else None

            invoice_number_match = re.search(r"номер накладной\s*(.*)$", text, re.IGNORECASE)
            invoice_number = invoice_number_match.group(1).strip() if invoice_number_match else None

            records.append({
                "sheet_name": sheet_name,
                "product_name": current_product,
                "product_code": current_product_code,
                "supplier": current_supplier,
                "invoice_line": text,
                "invoice_date": invoice_date,
                "invoice_number": invoice_number,
                "quantity_base_units": row.iloc[1] if len(row) > 1 else None,
                "return_qty": row.iloc[2] if len(row) > 2 else None,
                "total_qty": row.iloc[3] if len(row) > 3 else None,
                "price": row.iloc[4] if len(row) > 4 else None,
                "amount": row.iloc[5] if len(row) > 5 else None,
            })

    return pd.DataFrame(records)

# tüm sayfaları oku
xls = pd.ExcelFile(file_path)
all_results = []

for sheet in xls.sheet_names:
    df_raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
    parsed = parse_sheet(df_raw, sheet)
    all_results.append(parsed)

final_df = pd.concat(all_results, ignore_index=True)

print(final_df)

        sheet_name                             product_name product_code  \
0  Производитель 1  Продукт 1\nкод продукта - 100-00000-111         None   
1  Производитель 1  Продукт 1\nкод продукта - 100-00000-111         None   
2  Производитель 1  Продукт 1\nкод продукта - 100-00000-111         None   
3  Производитель 1  Продукт 2\nкод продукта - 200-00000-222         None   
4  Производитель 1  Продукт 3\nкод продукта - 300-00000-333         None   
5  Производитель 1  Продукт 3\nкод продукта - 300-00000-333         None   
6  Производитель 2  Продукт 1\nкод продукта - 100-00000-111         None   
7  Производитель 2  Продукт 2\nкод продукта - 200-00000-222         None   
8  Производитель 2  Продукт 3\nкод продукта - 300-00000-333         None   
9  Производитель 2  Продукт 3\nкод продукта - 300-00000-333         None   

                supplier                           invoice_line invoice_date  \
0  Поставщик 1 г. Москва  01/04/2025 номер накладной Н-11-00-11   01/04/2025   
1  

In [43]:
# önce string'e çevir (None vs. için güvenli)
final_df["product_name"] = final_df["product_name"].astype(str)

# product_code'u çıkar
final_df["product_code"] = final_df["product_name"].str.extract(
    r"код продукта\s*-\s*([\d-]+)"
)

# product_name'i temizle (sadece "Продукт X" kalsın)
final_df["product_name"] = final_df["product_name"].str.split("\n").str[0].str.strip()

In [46]:
# string güvenliği
final_df["supplier"] = final_df["supplier"].astype(str)

# city kolonunu çıkar
final_df["city"] = final_df["supplier"].str.extract(r"г\.\s*(.+)")

# supplier'ı temizle (şehir kısmını sil)
final_df["supplier"] = final_df["supplier"].str.replace(r"\s*г\..*", "", regex=True).str.strip()

In [49]:
# kolon silme
final_df = final_df.drop(columns=["invoice_line", "sheet_name"])

# kolon rename
final_df = final_df.rename(columns={
    "price": "price_per_unit",
    "amount": "total_sum"
})

In [53]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_name         10 non-null     object
 1   product_code         10 non-null     object
 2   supplier             10 non-null     object
 3   invoice_date         10 non-null     object
 4   invoice_number       10 non-null     object
 5   quantity_base_units  10 non-null     int64 
 6   return_qty           10 non-null     int64 
 7   total_qty            10 non-null     int64 
 8   price_per_unit       10 non-null     int64 
 9   total_sum            10 non-null     int64 
 10  city                 10 non-null     object
dtypes: int64(5), object(6)
memory usage: 1012.0+ bytes


In [54]:
final_df["invoice_date"] = pd.to_datetime(final_df["invoice_date"], dayfirst=True)

In [57]:
final_df = final_df.rename(columns={
    "product_name": "Номенклатура",
    "product_code": "Код продукта",
    "supplier": "Поставщик",
    "invoice_date": "Дата накладной",
    "invoice_number": "Номер накладной",
    "quantity_base_units": "Количество (в базовых единицах)",
    "return_qty": "Возврат",
    "total_qty": "Итог",
    "price_per_unit": "Цена за единицу",
    "total_sum": "Сумма",
    "city": "Город"
})

In [58]:
final_df.head(20)

,Номенклатура,Код продукта,Поставщик,Дата накладной,Номер накладной,Количество (в базовых единицах),Возврат,Итог,Цена за единицу,Сумма,Город
0,Продукт 1,100-00000-111,Поставщик 1,2025-04-01,Н-11-00-11,1,0,1,100,100,Москва
1,Продукт 1,100-00000-111,Поставщик 1,2025-04-01,Н-11-00-12,1,0,1,100,100,Москва
2,Продукт 1,100-00000-111,Поставщик 1,2025-04-01,Н-11-00-13,1,0,1,100,100,Москва
3,Продукт 2,200-00000-222,Поставщик 2,2025-04-02,Н-11-00-14,1,0,1,100,100,Москва
4,Продукт 3,300-00000-333,Поставщик 2,2025-04-03,Н-11-00-15,1,0,1,100,100,Москва
5,Продукт 3,300-00000-333,Поставщик 2,2025-04-03,,1,0,1,100,100,Москва
6,Продукт 1,100-00000-111,Поставщик 1,2025-04-01,Н-11-00-16,1,0,1,100,100,Москва
7,Продукт 2,200-00000-222,Поставщик 2,2025-04-02,Н-11-00-17,1,0,1,100,100,Москва
8,Продукт 3,300-00000-333,Поставщик 2,2025-04-03,Н-11-00-18,1,0,1,100,100,Москва
9,Продукт 3,300-00000-333,Поставщик 2,2025-04-03,Н-11-00-19,1,0,1,100,100,Москва


In [60]:
output_path = r"C:\Users\husnu\OneDrive\Desktop\interview\клиент_ромашка_продажи_clean.xlsx"
final_df.to_excel(output_path, index=False)